# Bi-Encoder 蒸餾(rbt6 → rbt3)— Colab GPU

把 6 層 bi-encoder 蒸成 3 層 student,前端 `bi_encoder_quant.onnx` ~57MB → ~38MB。

**先做**:Runtime → Change runtime type → **GPU**(T4 即可)。

**teacher 權重已遺失**,故 step 1 先重訓 rbt6 teacher,step 2 再蒸 rbt3。
規格與 gate 見 `docs/spec/bi-encoder-distill.md`。

## 0. 環境 — clone repo + 裝套件

In [ ]:
!nvidia-smi -L
%cd /content
![ -d nchu-edge-rental-ai ] || git clone https://github.com/eric20041027/nchu-edge-rental-ai.git
%cd /content/nchu-edge-rental-ai
!git pull
!pip -q install 'transformers<5' tokenizers onnx onnxruntime
# torch 已預裝於 Colab GPU runtime。確認版本:
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1. 重訓 rbt6 teacher(從 hfl/rbt6)

輸出 `saved_models/rbt6_bi_encoder/`。**記下這次 teacher 的 `recall_at_1_vs_pool`** 當蒸餾 gate 基準。

In [ ]:
!python -m pipeline.model_training.train_bi_encoder \
    --epochs 3 --batch-size 32 --lr 2e-5 --max-length 64 --bf16 --tf32

## 2. 蒸餾 rbt6 → rbt3 student

teacher-dir 預設就指向 step 1 的輸出。觀察 log:`Last step loss < First`、
`mean_student_teacher_cosine` ≳ 0.90(沒塌)、`recall_at_1_vs_pool` 不低於 step 1 太多。

> 想更靠 teacher 就提高 `--alpha`(預設 0.5);想更靠 ground-truth 就降低。

In [ ]:
!python -m pipeline.model_training.distill_bi_encoder \
    --epochs 3 --batch-size 32 --lr 3e-5 --alpha 0.5 --max-length 64 --bf16 --tf32

## 3. Export student → ONNX + INT8

輸出 `frontend/models/bi_encoder_dir/bi_encoder_quant.onnx`(目標 ~38MB)。pooling+L2-norm 在圖內。

In [ ]:
!python -m pipeline.model_training.export_bi_encoder \
    --saved-dir saved_models/rbt3_bi_encoder
!ls -la frontend/models/bi_encoder_dir/*.onnx

## 4. 用 student 重編 704 房源向量(關鍵,不可省)

student 是新向量空間,舊房源向量作廢。輸出 `frontend/assets/property_embeddings.json`(dim 仍 768)。

In [ ]:
!python -m pipeline.data_prep.build_property_embeddings \
    --saved-dir saved_models/rbt3_bi_encoder
!python -m pipeline.data_prep.build_property_embeddings --check

## 5. Recall gate — vector vs rule-based A/B

讀 step 3 的 onnx + step 4 的 property_embeddings(= 驗整條落地鏈)。
看 recall@K / NDCG@5 是否守住現役向量召回。

In [ ]:
!python tests/eval_vector_vs_rulebased.py

## 6. Gate 檢查 + 下載落地檔

**全過才落地**:

| 指標 | 門檻 |
|---|---|
| step2 loss 下降 | 必須 |
| step2 `mean_student_teacher_cosine` | ≳ 0.90 |
| step2 `recall_at_1_vs_pool` | ≥ step1 teacher − 0.02 |
| step5 recall@K / NDCG@5 | 不低於現役容忍內 |
| onnx 大小 | ≈ 38MB(↓ from 57MB) |

任一不過 → 不下載、不落地。下面 cell 把三個落地檔打包下載,本機解壓覆蓋後開 PR。

In [ ]:
import os
sz = os.path.getsize('frontend/models/bi_encoder_dir/bi_encoder_quant.onnx') / 1e6
print(f'bi_encoder_quant.onnx = {sz:.1f} MB  (目標 ≈38, 原 ~57)')
assert sz < 50, '體積沒降到預期,檢查 student 層數'

# 三個落地檔一起打包(onnx + 重編的房源向量,版本必須同批)
!cd /content/nchu-edge-rental-ai && tar czf /content/bi_encoder_rbt3_artifacts.tgz \
    frontend/models/bi_encoder_dir/bi_encoder_quant.onnx \
    frontend/models/bi_encoder_dir/bi_encoder.onnx \
    frontend/assets/property_embeddings.json
from google.colab import files
files.download('/content/bi_encoder_rbt3_artifacts.tgz')

## 落地(本機)

```bash
cd <repo>
tar xzf ~/Downloads/bi_encoder_rbt3_artifacts.tgz   # 覆蓋三檔
git checkout -b claude/bi-encoder-rbt3-land
git add frontend/models/bi_encoder_dir/ frontend/assets/property_embeddings.json
git commit -m 'feat(model): 落地 rbt3 bi-encoder(57→38MB)+ 重編房源向量'
```

⚠️ onnx 與 property_embeddings **必須同一次蒸餾產出**,版本錯配 = 召回壞掉。
前端冷載入應隨 bi-encoder onnx 縮小而下降(用 benchmark.html 驗)。